# Lab 12 — HR Attrition Mini Case Study
**Course Capstone Track** · Intermediate Capstone · ~75 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute overall attrition and slice by Department / OverTime
2. Bin income into quartiles and compare attrition rates
3. Spot confounders and non-monotonic signals
4. Write 3 actionable hypotheses into attrition_report.md

## Datasets (this folder)
- `HR-Employee-Attrition-synth.csv` — auto-download from `https://raw.githubusercontent.com/aaubs/ds-master/main/apps/M1-attrition-streamlit/HR-Employee-Attrition-synth.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-12-hr-attrition-capstone/lab-12-hr-attrition-capstone.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`HR-Employee-Attrition-synth.csv`, `emp_attrition.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-12-hr-attrition-capstone"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-12-hr-attrition-capstone/bundle/dataset.zip"
NEED = ["HR-Employee-Attrition-synth.csv", "emp_attrition.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Course Capstone Track: EDA → Insight → Executive One-Pager

> **Scenario:** People analytics shares `HR-Employee-Attrition-synth.csv` (2000 rows, IBM-style HR schema, 35 columns). Deliver an executive one-pager: overall attrition rate, rates by Department and OverTime, income quartiles vs attrition, and **3 actionable hypotheses** with supporting numbers. Write `attrition_report.md`.
>
> **You will learn:** end-to-end pipeline, EDA → insight → narrative, avoiding spurious correlations.
> **Time:** ~75 minutes. **Level:** Intermediate Capstone. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

This capstone is the whole course compressed into one deliverable: frame the question, slice the data, associate carefully, interpret honestly, and propose actions you could actually test. The output is not a notebook for its own sake — it is `attrition_report.md`, a one-pager a busy executive can read in two minutes and still see the numbers, the caveats, and the next experiments.

### Capstone mental map

Work top-to-bottom: you cannot say “where attrition is worst” before you’ve defined the baseline rate, and you cannot propose interventions before you’ve checked whether your “driver” is just a proxy for something else. Each stage produces a small artifact that feeds the next.

| Stage | Question | Artifact |
|---|---|---|
| Frame | What is attrition rate? | headline number |
| Slice | Where is it worst? | Dept / OverTime tables |
| Associate | Income vs leaving? | quartile rates + chart |
| Interpret | Causation? confounders? | caveats section |
| Act | What would we try? | 3 hypotheses |

> **Pitfall:** this file is **synthetic**. Numbers here (including the counter-intuitive income pattern below) describe a generated population, not a real labour market. Report what the file says, flag the synthetic provenance in your caveats, and never present these rates as industry benchmarks.

---

### 1. Load and profile (local first, Colab fallback)

Why profiling comes first: before any slice or chart, you need the denominator and the baseline. The overall attrition rate (**0.149**, i.e. 298 leavers of 2000) is the number every later percentage is compared against — a department rate of 16% means something different if the company baseline is 5% versus 15%. Department counts (R&D 1098, Sales 799, HR 103) also warn you upfront which slices will be noisy.

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load_hr():
    local = "HR-Employee-Attrition-synth.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/aaubs/ds-master/main/apps/"
            "M1-attrition-streamlit/HR-Employee-Attrition-synth.csv",
            local,
        )
    df = pd.read_csv(local)
    if str(df.columns[0]).startswith("Unnamed"):
        df = df.drop(columns=df.columns[0])
    return df

df = load_hr()
print(df.shape)          # (2000, 35)
print(df["Attrition"].value_counts())
# No 1702, Yes 298
print("overall attrition rate:", round((df["Attrition"] == "Yes").mean(), 4))
# 0.149
print(df["Department"].value_counts())
# Research & Development 1098, Sales 799, Human Resources 103
print("mean age:", df["Age"].mean().round(2))  # 37.24


**What to notice:** shape is **(2000, 35)**; leavers are **298 Yes vs 1702 No**, giving overall rate **0.149**; department sizes are heavily skewed (**1098 / 799 / 103**), so the HR-department rate will rest on only ~103 people; mean age **37.24** frames the population. The `Unnamed` column drop guards against a stray index column from the CSV export — always check `df.shape` matches your expectation after load.

---

### 2. Headline: attrition by Department and OverTime

Why these two slices lead the one-pager: Department and OverTime are the cuts an executive will ask for first (“where hurts?” and “is it the overtime?”). The `rate()` helper computes `(s == "Yes").mean()` per group and sorts descending, so the worst group is always the top row — readable at a glance and easy to paste into the report table.

In [ ]:
def rate(col):
    return (df.groupby(col)["Attrition"]
              .apply(lambda s: (s == "Yes").mean())
              .sort_values(ascending=False)
              .round(4))

print(rate("Department"))
# Research & Development    0.1621
# Sales                     0.1377
# Human Resources           0.0971

print(rate("OverTime"))
# Yes    0.1869
# No     0.1387


**What to notice:** R&D leads departments at **0.1621** while HR is lowest at **0.0971** — but HR’s n=103 makes that 9.7% fragile (a handful of leavers moves it several points). The OverTime split is **0.1869 vs 0.1387**: staff on overtime leave at a clearly higher rate than those who aren’t.

**OverTime gap:** 18.7% vs 13.9% — **+4.8 pp** for staff working overtime.

In [ ]:
ax = rate("OverTime").plot(kind="bar", figsize=(5, 3), rot=0, ylim=(0, 0.25),
                           title="Attrition rate by OverTime")
ax.set_ylabel("Attrition rate")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
fig = ax.get_figure(); fig.tight_layout(); fig.savefig("attrition_overtime.png", dpi=120)


**What to notice:** the y-axis runs **0 to 0.25** with percent formatting, so the two bars compare from a true zero baseline; the saved file is `attrition_overtime.png`. Keep this chart next to its n-counts in the report — a rate without a denominator invites over-reading.

---

### 3. Income quartiles vs attrition

Why quartiles instead of a raw scatter: income is skewed (edges run 1009 to 28723), so binning into quartiles gives four roughly equal groups and four rates a reader can compare directly. This is the section most likely to produce a *counter-intuitive* result — which is exactly why it earns its own caveat rather than being quietly dropped.

In [ ]:
df["IncomeQ"] = pd.qcut(df["MonthlyIncome"], 4,
                        labels=["Q1", "Q2", "Q3", "Q4"])
edges = df["MonthlyIncome"].quantile([0, .25, .5, .75, 1]).tolist()
print("quartile edges:", [round(e, 2) for e in edges])
# [1009.0, 5040.75, 6776.0, 9667.25, 28723.0]

by_q = df.groupby("IncomeQ", observed=True)["Attrition"].apply(
    lambda s: (s == "Yes").mean()).round(4)
print(by_q)
# Q1 0.1440
# Q2 0.1238
# Q3 0.1503
# Q4 0.1780


**What to notice:** quartile edges are **1009, 5040.75, 6776, 9667.25, 28723** — note the long right tail (Q4 spans 9667→28723 while Q1 spans 1009→5040). Rates are **Q1 0.1440 · Q2 0.1238 · Q3 0.1503 · Q4 0.1780**: not monotonic, and the *highest* quartile leaves most.

**Counter-intuitive (synthetic) pattern:** highest quartile has the *highest* attrition (17.8%) — in this synthetic file, not the classic IBM original. Report what you see; don’t force the textbook story.

In [ ]:
ax = by_q.plot(kind="bar", figsize=(5, 3), rot=0, ylim=(0, 0.25),
               title="Attrition rate by income quartile")
ax.set_ylabel("Attrition rate")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
for i, v in enumerate(by_q):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=9)
fig = ax.get_figure(); fig.tight_layout()
fig.savefig("attrition_income_quartiles.png", dpi=120)
print("saved charts")


**What to notice:** each bar is labelled with its own percentage (`14.4%`, `12.4%`, `15.0%`, `17.8%`), so the chart is readable without a tooltip; the figure saves as `attrition_income_quartiles.png`. The Q4-over-Q2 gap (**0.0542**, 5.4 pp) is the number to quote — and to caveat as synthetic.

---

### 4. Cross-checks (confounders)

Why cross-check before concluding: a single two-way rate can always be a proxy. Overtime might correlate with income; job satisfaction might correlate with department; leavers here actually earn *more* on average than stayers — all signals that any “X causes attrition” claim needs a confounder pass first. Run these three checks on every driver you plan to put in the one-pager.

In [ ]:
# Overtime × income — is the OT gap just a pay effect?
print(pd.crosstab(df["OverTime"], df["IncomeQ"], normalize="index").round(3))

# Job satisfaction vs attrition (weak in this synth?)
print(rate("JobSatisfaction"))
# 4: 0.1576, 2: 0.1565, 1: 0.1383, 3: 0.1254  (non-monotonic — weak signal)

# Mean income by leaver status
print(df.groupby("Attrition")["MonthlyIncome"].mean().round(2))
# No 8033.23, Yes 8676.02  (leavers earn slightly more here)


**What to notice:** JobSatisfaction rates (**4: 0.1576, 2: 0.1565, 1: 0.1383, 3: 0.1254**) are non-monotonic — the “happier employees stay” story isn’t supported here; mean income shows leavers at **8676.02** vs stayers **8033.23**, consistent with the synthetic Q4 pattern rather than the classic one. The OverTime×Income crosstab tells you whether the OT gap survives within pay bands.

> **Spurious-correlation guard:** always ask “what else varies with this feature?” Department, job level, and overtime often proxy for each other. n = 103 in HR dept → wide uncertainty on that 9.7% rate.

> **Pitfall:** association ≠ causation, especially on observational HR data. Before writing “overtime causes attrition,” check overlapping slices (JobLevel, Department, income band) and remember small cells (HR n=103) have wide, easily-overstated confidence.

---

### 5. Draft the executive one-pager

Why write the report *as code*: an f-string report regenerates from the dataframe, so the headline rate, gaps, and counts can’t drift out of sync with the analysis if the data reloads. The structure below is the executive frame — headline number first, then where it concentrates, then three testable hypotheses (not conclusions), then caveats. Executives read top-down; put the 14.9% where they can’t miss it.

In [ ]:
report = f"""# Attrition one-pager — HR synthetic (n=2000)

## Headline
- Overall attrition rate: **{(df['Attrition']=='Yes').mean():.1%}** (298 leavers / 2000).
- Mean age {df['Age'].mean():.1f}; majority in R&D ({(df['Department']=='Research & Development').mean():.0%}).

## Where attrition concentrates
| Cut | Higher rate | Lower rate | Gap |
|---|---|---|---|
| OverTime | Yes {(df.loc[df.OverTime=='Yes','Attrition']=='Yes').mean():.1%} | No {(df.loc[df.OverTime=='No','Attrition']=='Yes').mean():.1%} | +4.8 pp |
| Department | R&D 16.2% | HR 9.7% (n=103, noisy) | — |
| Income quartile | Q4 17.8% | Q2 12.4% | +5.4 pp |

## Three hypotheses (to test, not truths)
1. **Overtime load raises exit risk.** Design: compare matched OT/non-OT pairs within JobLevel; pilot workload cap in R&D.
2. **High earners in this population leave for external offers** (Q4 17.8%). Design: exit-interview coding + comp-ratio analysis vs market.
3. **R&D career-path opacity drives exits** (dept rate 16.2%). Design: promotion velocity by tenure; mentoring experiment in two teams.

## Caveats
- Synthetic data; JobSatisfaction signal is non-monotonic — do not over-interpret.
- Department rates with n≈100 have wide CIs; show counts beside every percentage.
- Association ≠ causation; validate with interventions before policy change.
"""
open("attrition_report.md", "w", encoding="utf-8").write(report)
print("wrote attrition_report.md")
print(report[:400])


**What to notice:** the file writes to `attrition_report.md` and the preview confirms the headline (overall rate formatted from the dataframe, 298/2000) plus the concentration table; each hypothesis carries a rate (18.7% vs 13.9%, Q4 17.8%, R&D 16.2%) *and* a validation design, and the caveats section names the synthetic provenance and the n≈100 warning explicitly.

---

### What good looks like (rubric)

Before Exercises, calibrate against the checklist an reviewer would use on your `attrition_report.md`:

- **Headline present and correct:** overall attrition printed to one decimal (14.9%) with its counts (298/2000), derived from the loaded frame — not copied by hand.
- **Slices with denominators:** Department and OverTime rates each shown beside their n (especially HR n=103 flagged as noisy).
- **Income section tells the truth:** Q1–Q4 rates match the file (0.1440 / 0.1238 / 0.1503 / 0.1780), the counter-intuitive Q4 result is reported rather than smoothed over, and “synthetic” appears in the caveat.
- **Confounder pass visible:** at least the OverTime×Income crosstab and the non-monotonic JobSatisfaction rates are acknowledged before any causal-sounding claim.
- **Three hypotheses, each testable:** every hypothesis cites at least one computed number and one validation design (matched pairs, pilot, exit-interview coding, etc.).
- **Caveats section non-empty:** synthetic data, small-n slices, and association-vs-causation all stated.
- **Artifacts exist:** `attrition_report.md` plus `attrition_overtime.png` and `attrition_income_quartiles.png`.

A report missing any of these reads as a vibe, not an analysis — fix the gap rather than dressing the prose.

---

## Exercises (do these!)

### Exercise 1 — Attrition by OverTime
Print attrition rate for `OverTime == "Yes"` and `"No"` (4 d.p.). What is the percentage-point gap?
*Expected: Yes 0.1869 · No 0.1387 · gap ≈ 0.0482 (4.8 pp).*

**Follow-up:** How many leavers is that in absolute terms? Check: 298.

<details>
<summary>Hint</summary>

`df.groupby("OverTime")["Attrition"].apply(lambda s: (s=="Yes").mean())`.
</details>

### Exercise 2 — Income quartiles vs attrition
`pd.qcut(MonthlyIncome, 4)` → attrition rate per quartile. Print the 4 rates and the quartile edges.
*Expected: Q1 0.1440 · Q2 0.1238 · Q3 0.1503 · Q4 0.1780 · edges 1009, 5040.75, 6776, 9667.25, 28723.*

**Follow-up:** How large is the Q4 minus Q2 attrition gap? Check: 0.0542.

<details>
<summary>Hint</summary>

`observed=True` on groupby avoids empty-group warnings in newer pandas.
</details>

### Exercise 3 — Draft 3 hypotheses with numbers
Write three hypotheses; each must cite at least one number from this lab and one validation step. Save under `attrition_report.md`.
*Expected: free-form — e.g. OT pilot, R&D career path, comp-ratio for Q4 leavers — each with rates/n from Sections 2–4.*

**Follow-up:** How many R and D leavers? Check: 178.

<details>
<summary>Hint</summary>

Template: “If we [intervention], then [metric] changes, because [evidence: X% vs Y%]; validate via [design].”
</details>

---

## Solutions

Worked answers for the three exercises and follow-ups. Try each exercise before revealing these; the asserts double-check your own counts.

In [ ]:
# --- Solution 1 ---
rates = df.groupby("OverTime")["Attrition"].apply(lambda s: (s == "Yes").mean())
print(rates.round(4))
gap = rates["Yes"] - rates["No"]
print(f"gap = {gap:.4f} ({gap*100:.1f} pp)")
# Yes    0.1869
# No     0.1387
# gap = 0.0482 (4.8 pp)

# --- Solution 2 ---
df["IncomeQ"] = pd.qcut(df["MonthlyIncome"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df.groupby("IncomeQ", observed=True)["Attrition"]
      .apply(lambda s: (s == "Yes").mean()).round(4))
print(df["MonthlyIncome"].quantile([0, .25, .5, .75, 1]).tolist())
# Q1 0.1440  Q2 0.1238  Q3 0.1503  Q4 0.1780
# [1009.0, 5040.75, 6776.0, 9667.25, 28723.0]

# --- Solution 3 ---
# Template (fill with your own numbers from above):
hypotheses = """
1. Reducing mandatory overtime in R&D will cut attrition — OT yes 18.7% vs no 13.9%.
2. Q4 earners need external-market comp review — Q4 attrition 17.8% vs Q2 12.4%.
3. Promotion clarity in R&D should be audited — dept rate 16.2% (n=1098).
Validate: matched cohorts / pilot teams / exit-interview themes; not one-shot correlations.
"""
print(hypotheses)
# (Section 5 already writes attrition_report.md with this structure)

# --- Follow-up 1 ---
n_leavers = int((df["Attrition"] == "Yes").sum())
print(n_leavers)  # 298
assert n_leavers == 298

# --- Follow-up 2 ---
bq = df.groupby("IncomeQ", observed=True)["Attrition"].apply(lambda s: (s == "Yes").mean()).round(4)
qgap = round(bq["Q4"] - bq["Q2"], 4)
print(qgap)  # 0.0542
assert qgap == 0.0542

# --- Follow-up 3 ---
n_rd = int(((df["Department"] == "Research & Development") & (df["Attrition"] == "Yes")).sum())
print(n_rd)  # 178
assert n_rd == 178


**What to notice:** the three asserts pin the lab’s anchors — **298** total leavers, **0.0542** Q4-minus-Q2 gap, and **178** R&D leavers (so R&D contributes 178 of the 298 exits, matching its 16.2% rate on n=1098). If your Q4-minus-Q2 gap differs, re-check the quartile edges and `observed=True` from Section 3.

### What to learn next
- Logistic regression / SHAP for driver ranking (Course 2).
- Survival analysis for time-to-exit.
- Fairness: would an OT-based policy disproportionately hit caregiver demographics?
- Cheat sheet: headline rate → slice → quartile/associate → confounders → hypotheses with validation plans.

*Files in this folder: `HR-Employee-Attrition-synth.csv` (primary), `emp_attrition.csv` (IBM alternate, n=1470) · outputs `attrition_overtime.png`, `attrition_income_quartiles.png`, `attrition_report.md`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
